In [1]:
seed = 1874033229

In [2]:
import os

random_data = os.urandom(4)
# seed = int.from_bytes(random_data, byteorder="big")
print(f"Chosen Unbiased Seed: {seed}")

Chosen Unbiased Seed: 1874033229


In [3]:
import random
import numpy as np
import torch

def set_seed(seed: int):
    # ---- Python RNG ----
    random.seed(seed)

    # ---- NumPy RNG ----
    np.random.seed(seed)

    # ---- PyTorch CPU/GPU RNG ----
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"Seed set to: {seed}")

set_seed(seed)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

Seed set to: 1874033229


/gpu-data2/kfot/miniconda3/envs/myenv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import copy
import matplotlib.pyplot as plt

os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [5]:
# Set device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [15]:
def flatten_tensors(tensors):
    return torch.cat([t.reshape(-1) for t in tensors])

def unflatten_like(flat, shapes):
    out = []
    idx = 0
    for s in shapes:
        numel = torch.tensor(s).prod().item()
        out.append(flat[idx:idx + numel].reshape(s))
        idx += numel
    return out

class MaskHandler:
    def __init__(self, model=None):
        self.param_to_mask = {}
        self.name_to_mask = {}

        if model is not None:
            self.register_model(model)

    def register_model(self, model):
        self.param_to_mask = {}
        self.name_to_mask = {}
        for name, p in model.named_parameters():
            if "fc" in name:
                continue
            mask = torch.ones_like(p)
            self.param_to_mask[id(p)] = mask
            self.name_to_mask[name] = mask

    def export(self):
        return {k: v.clone() for k, v in self.name_to_mask.items()}

    def load(self, model, mask_dict):
        self.param_to_mask = {}
        self.name_to_mask = {}

        for name, p in model.named_parameters():
            if name in mask_dict:
                m = mask_dict[name].to(p.device)
                self.param_to_mask[id(p)] = m
                self.name_to_mask[name] = m

        self.attach_model(model)
        
    def attach_model(self, model):
        for m in model.modules():
            if hasattr(m, "apply_mask"):
                m.mask_handler = self
        model.mask_handler = self

    def get(self, p):
        return self.param_to_mask.get(id(p), torch.ones_like(p))
    
    def set(self, p, m):
        self.param_to_mask[id(p)] = m

class Pruner:
    def __init__(self, model):
        self.model = model

        self.params = []
        for name, p in model.named_parameters():
            if "fc" in name:
                continue
            if p.requires_grad:
                self.params.append((name, p))

        self.shapes = [p.shape for _, p in self.params]

        self.mask_handler = MaskHandler(self.model)
    
    def compute_scores_snip(self, dataloader, num_batches=10):
        self.model.train()

        # SNIP. We aggregate scores instead of gradients! 
        # Initial network -> untrained -> gradient over lots of batches ~= 0 (cancellation)
        # But, ~0 across all batches -> unimportant.
        # Variety across batches (and then cancellation) -> important. 
        scores = [torch.zeros_like(p) for _, p in self.params]

        for i, (x, y) in enumerate(dataloader):
            if i >= num_batches:
                break

            self.model.zero_grad()

            x, y = x.to(device), y.to(device)
            out = self.model(x)
            loss = F.cross_entropy(out, y)

            loss.backward()

            # SNIP criterion, note: |grad_c(c*w)| = |grad_w(w) * w|
            for j, (_, p) in enumerate(self.params):
                if p.grad is not None:
                    scores[j] += torch.abs(p.grad * p)

        return flatten_tensors(scores)
    
    def compute_scores_snip2(self, dataloader, num_batches=10):
        self.model.train()

        scores = [torch.zeros_like(p) for _, p in self.params]

        for i, (x, y) in enumerate(dataloader):
            if i >= num_batches:
                break

            self.model.zero_grad()

            x, y = x.to(device), y.to(device)
            out = self.model(x)
            loss = F.cross_entropy(out, y)

            loss.backward()

            # SNIP criterion, note: |grad_c(c*w)| = |grad_w(w) * w|
            for j, (name, p) in enumerate(self.params):
                if "linact" in name:
                    scores[j] = torch.full_like(p, float("inf"))
                if p.grad is not None:
                    scores[j] += torch.abs(p.grad * p)

        return flatten_tensors(scores)
    
    def compute_scores_grasp(self, dataloader, num_batches=2):
        self.model.train()

        # GraSP criterion, -(theta) * H g, where theta: parameters, H: Hessian, g: gradient
        # Equivalently: si = -wi \partial_i(g^T g)
        # (Note: \partial_i(g^T) g = 1/2 \partial_i(g^T g). constant 1/2 is irrelevant for order)

        # ===== accumulate gradient g =====
        grad_w = None

        for i, (x, y) in enumerate(dataloader):
            if i >= num_batches:
                break

            x, y = x.to(device), y.to(device)

            out = self.model(x)
            loss = F.cross_entropy(out, y)

            grads = torch.autograd.grad(loss, [p for _, p in self.params],
                                        create_graph=True)

            if grad_w is None:
                grad_w = grads
            else:
                grad_w = [g1 + g2 for g1, g2 in zip(grad_w, grads)]

        # ===== compute g^T g =====
        grad_inner = 0
        for g in grad_w:
            grad_inner += (g * g).sum()

        # ===== second backward pass for \partial(g^T g)=====
        self.model.zero_grad()

        grad_inner.backward()

        # ===== GraSP scores =====
        scores = []
        for (name, p), g in zip(self.params, grad_w):
            if p.grad is None:
                scores.append(torch.zeros_like(p))
            else:
                # Note the negative sign
                scores.append(-p * p.grad)

        return flatten_tensors(scores)
    
    def get_mask(self, scores, sparsity):
        k = int((1 - sparsity) * scores.numel())

        _, idx = torch.topk(scores, k=k, largest=True, sorted=False)

        mask = torch.zeros_like(scores, dtype=torch.bool)
        mask[idx] = True

        return mask
    
    def set_mask(self, flat_mask):
        masks = unflatten_like(flat_mask, self.shapes)

        with torch.no_grad():
            for (name, p), m in zip(self.params, masks):
                m = m.to(p.device)
                self.mask_handler.set(p, m)
                self.mask_handler.name_to_mask[name] = m

        self.mask_handler.attach_model(self.model)

    def prune(self, dataloader, sparsity=0.5, num_batches=10, method="snip"):
        self.model.set_state("pruning")
        if method == "snip":
            scores = self.compute_scores_snip(dataloader, num_batches)
        elif method == "snip2":
            scores = self.compute_scores_snip2(dataloader, num_batches)
        else:
            scores = self.compute_scores_grasp(dataloader, num_batches)

        mask = self.get_mask(scores, sparsity)

        self.set_mask(mask)
        
        self.model.set_state("normal")

        return self.model.count_params()

from torch.utils.cpp_extension import load
# Load and compile CUDA extension
maxplus_conv2d = load(
    name="maxplus_conv2d",
    sources=["../conv2d_cuda.cu"],
    # extra_cuda_cflags=["-lineinfo"],  # Optional: Debugging info
    verbose=True
)

from torch.autograd import Function    
class MaxPlusConv2dFunction(Function):
    @staticmethod
    def forward(ctx, input, weight, bias, stride, padding):
        output, argmax_input_idx, argmax_weight_idx = maxplus_conv2d.maxplus_conv2d_forward(
            input, weight, bias, stride, padding
        )
        ctx.save_for_backward(argmax_input_idx, argmax_weight_idx)
        ctx.input_shape = input.shape
        ctx.weight_shape = weight.shape
        ctx.stride = stride
        ctx.padding = padding
        return output

    @staticmethod
    def backward(ctx, grad_output):
        argmax_input_idx, argmax_weight_idx = ctx.saved_tensors
        B, C_in, H_in, W_in = ctx.input_shape
        C_out, _, K, _ = ctx.weight_shape
        stride = ctx.stride
        padding = ctx.padding
        H_out = (H_in + 2 * padding - K) // stride + 1
        W_out = (W_in + 2 * padding - K) // stride + 1

        grad_input, grad_weight, grad_bias = maxplus_conv2d.maxplus_conv2d_backward(
            grad_output, argmax_input_idx, argmax_weight_idx,
            B, C_in, C_out, H_in, W_in, H_out, W_out, K
        )
        return grad_input, grad_weight, grad_bias, None, None

# Convenience wrapper
def maxplus_conv2d_wrapper(input, weight, bias=None, stride=1, padding=0):
    return MaxPlusConv2dFunction.apply(input, weight, bias, stride, padding)

# class MorphConv2d(nn.Module):
#     def __init__(self, in_channels, out_channels, kernel_size=(3,3), stride=(1,1), padding=(1,1), bias=True, alpha=1.0):
#         super(MorphConv2d, self).__init__()
#         self.in_channels = in_channels
#         self.out_channels = out_channels
#         self.kernel_size = kernel_size
#         self.stride = stride
#         self.padding = padding
#         self.bias = bias
#         self.alpha = alpha

#         self.weight = nn.Parameter(
#             torch.normal(mean=0.0, std=alpha, size=(out_channels, in_channels, kernel_size[0], kernel_size[1]))
#         )
#         if bias:
#             self.b = nn.Parameter(torch.normal(mean=0.0, std=1.0, size=(out_channels, )))
#             self.b2 = nn.Parameter(torch.normal(mean=0.0, std=1.0, size=(out_channels, )))
#         else:
#             self.register_buffer("b", torch.full((out_channels,), float("-inf")))
#             self.register_buffer("b2", torch.full((out_channels,), float("inf")))
#             # self.b = nn.Parameter(torch.full((out_channels, ), float('-inf')))
#             # self.b2 = nn.Parameter(torch.full((out_channels, ), float('inf')))

#         self.mask_handler = None

#         self.state = "normal"

#     def apply_mask(self, param, neg = False):
#         mask = self.mask_handler.get(param) if self.mask_handler is not None else torch.ones_like(param)
#         if neg:  # for min: masked values should be +inf
#             return torch.where(mask.bool(), param, torch.full_like(param, float('inf'))), mask
#         else:    # for max: masked values should be -inf
#             return torch.where(mask.bool(), param, torch.full_like(param, float('-inf'))), mask

#     def forward(self, x):
#         # Max morphological operation
#         max_w, _ = self.apply_mask(self.weight)
#         min_w, _ = self.apply_mask(self.weight, True)
#         b = self.b
#         b2 = self.b2
#         if self.bias:
#             b, _ = self.apply_mask(self.b)
#             b2, _ = self.apply_mask(self.b2, True)
 
#         x_max = maxplus_conv2d_wrapper(x, max_w, b, self.stride[0], self.padding)

#         # Min morphological operation
#         x_min = maxplus_conv2d_wrapper(-x, -min_w, -b2, self.stride[0], self.padding)
#         x_min = -x_min

#         # Aggregation of max and min operations
#         x_out = (x_max + x_min)/2

#         return x_out

class MorphConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=(3,3), stride=(1,1), padding=(1,1), bias=True, alpha=1.0):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.bias = bias

        self.weight = nn.Parameter(
            torch.normal(mean=0.0, std=alpha, size=(out_channels, in_channels, kernel_size[0], kernel_size[1]))
        )

        if bias:
            self.b = nn.Parameter(torch.normal(mean=0.0, std=1.0, size=(out_channels,)))
            self.b2 = nn.Parameter(torch.normal(mean=0.0, std=1.0, size=(out_channels,)))
        else:
            self.register_buffer("b", torch.full((out_channels,), float("-inf")))
            self.register_buffer("b2", torch.full((out_channels,), float("inf")))

        self.mask_handler = None

    def get_mask(self, param):
        if self.mask_handler is None:
            return torch.ones_like(param, dtype=torch.bool)
        return self.mask_handler.get(param).bool()

    def apply_mask(self, param, neg=False):
        mask = self.get_mask(param)
        fill = float("inf") if neg else float("-inf")
        return torch.where(mask, param, torch.full_like(param, fill)), mask

    def propagate_valid(self, valid, weight_mask):
        # valid: [B, C_in, H, W], weight_mask: [C_out, C_in, K, K]
        count = F.conv2d(
            valid.float(),
            weight_mask.float(),
            bias=None,
            stride=self.stride,
            padding=self.padding
        )
        return count > 0

    def forward(self, inp):
        if isinstance(inp, tuple):
            x, valid = inp
        else:
            x = inp
            valid = torch.ones_like(x, dtype=torch.bool)

        max_w, mask_w = self.apply_mask(self.weight, neg=False)
        min_w, _ = self.apply_mask(self.weight, neg=True)

        b = self.b
        b2 = self.b2
        if self.bias:
            b, mask_b = self.apply_mask(self.b, neg=False)
            b2, mask_b2 = self.apply_mask(self.b2, neg=True)

        x_max = torch.where(valid, x, torch.full_like(x, float("-inf")))
        x_min = torch.where(valid, x, torch.full_like(x, float("inf")))
        x_max = maxplus_conv2d_wrapper(x_max, max_w, b, self.stride[0], self.padding)
        x_min = maxplus_conv2d_wrapper(-x_min, -min_w, -b2, self.stride[0], self.padding)
        x_min = -x_min

        out = (x_max + x_min) / 2

        valid_out = self.propagate_valid(valid, mask_w)

        if self.bias:
            valid_bias = (mask_b & mask_b2).view(1, -1, 1, 1)
            valid_out = valid_out | valid_bias

        out = torch.where(valid_out, out, torch.zeros_like(out))
        assert not torch.isnan(out).any()
        return out, valid_out
    
# class ConvLinAct(nn.Module):
#     def __init__(self, channels, method="simple"):
#         super(ConvLinAct, self).__init__()
#         self.method = method

#         self.a = nn.Parameter(torch.zeros(3, 3, channels))
#         self.a.data[1,1,:] += 1

#         self.mask_handler = None

#     def apply_mask(self, param):
#         mask = self.mask_handler.get(param) if self.mask_handler is not None else torch.ones_like(param)
#         return param * mask, mask

#     def forward(self, x):
#         masked_a, _ = self.apply_mask(self.a)
#         tmp = torch.diag_embed(masked_a)
#         tmp = torch.transpose(tmp, 0, 2)
#         tmp = torch.transpose(tmp, 1, 3)
#         x = nn.functional.conv2d(x, tmp, padding=1)
#         return x

class ConvLinAct(nn.Module):
    def __init__(self, channels, method="simple"):
        super().__init__()
        self.method = method
        self.a = nn.Parameter(torch.zeros(3, 3, channels))
        self.a.data[1, 1, :] += 1
        self.mask_handler = None

    def get_mask(self, param):
        if self.mask_handler is None:
            return torch.ones_like(param, dtype=torch.bool)
        return self.mask_handler.get(param).bool()

    def apply_mask(self, param):
        mask = self.get_mask(param)
        return param * mask, mask

    def forward(self, inp):
        if isinstance(inp, tuple):
            x, valid = inp
        else:
            x = inp
            valid = torch.ones_like(x, dtype=torch.bool)

        masked_a, mask_a = self.apply_mask(self.a)

        tmp = torch.diag_embed(masked_a)
        tmp = torch.transpose(tmp, 0, 2)
        tmp = torch.transpose(tmp, 1, 3)
        out = F.conv2d(x, tmp, padding=1)

        mask_tmp = torch.diag_embed(mask_a.float())
        mask_tmp = torch.transpose(mask_tmp, 0, 2)
        mask_tmp = torch.transpose(mask_tmp, 1, 3)
        valid_out = F.conv2d(valid.float(), mask_tmp, padding=1) > 0

        out = torch.where(valid_out, out, torch.zeros_like(out))
        return out, valid_out

class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, kernel_size=1, stride=1, bias=False)

        self.mask_handler = None

    def apply_mask(self, module, x):
        if isinstance(module, nn.Conv2d):
            weight = module.weight * (self.mask_handler.get(module.weight) if self.mask_handler is not None else torch.ones_like(module.weight))
            bias = None
            if module.bias is not None:
                bias = module.bias * (self.mask_handler.get(module.bias) if self.mask_handler is not None else torch.ones_like(module.bias))
            return F.conv2d(
                x, weight, bias,
                module.stride, module.padding,
                module.dilation, module.groups
            )

        elif isinstance(module, nn.Linear):
            weight = module.weight * (self.mask_handler.get(module.weight) if self.mask_handler is not None else torch.ones_like(module.weight))
            bias = None
            if module.bias is not None:
                bias = module.bias * (self.mask_handler.get(module.bias) if self.mask_handler is not None else torch.ones_like(module.bias))
            return F.linear(x, weight, bias)

        elif isinstance(module, nn.BatchNorm2d):
            weight = module.weight * (self.mask_handler.get(module.weight) if self.mask_handler is not None else torch.ones_like(module.weight))
            bias = None
            if module.bias is not None:
                bias = module.bias * (self.mask_handler.get(module.bias) if self.mask_handler is not None else torch.ones_like(module.bias))
            return F.batch_norm(x, module.running_mean, module.running_var, weight, bias, self.training, module.momentum)

        else:
            return module(x)

    def forward(self, x):
        out = F.relu(self.apply_mask(self.bn1, self.apply_mask(self.conv1, x)))
        out = self.apply_mask(self.bn2, self.apply_mask(self.conv2, out))
        out += self.apply_mask(self.shortcut, x)
        return F.relu(out)

class ResNet20(nn.Module):
    def __init__(self, in_channels=3, num_blocks=[3, 3, 3], num_classes=10):
        super().__init__()
        self.in_planes = 16
        self.conv1 = nn.Conv2d(in_channels, 16, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        self.layer1 = self._make_layer(16, num_blocks[0])
        self.pool1 = nn.MaxPool2d(2, 2)

        self.layer2 = self._make_layer(32, num_blocks[1])
        self.pool2 = nn.MaxPool2d(2, 2)

        self.layer3 = self._make_layer(64, num_blocks[2])
        
        self.fc = nn.Linear(64, num_classes)

        self.mask_handler = None

        self.state = "normal"

    def set_state(self, state):
        self.state = state

    def count_params(self):
        total = sum(p.numel() for p in self.parameters())
        active = 0
        for name, p in self.named_parameters():
            mask = self.mask_handler.get(p) if self.mask_handler is not None else torch.ones_like(p)
            print(f"{name}: Initial params = {p.numel()}, Post-pruning active params = {mask.sum().item()}")
            active += mask.sum().item()
        return total, active
    
    def apply_mask(self, module, x):
        if isinstance(module, nn.Conv2d):
            weight = module.weight * (self.mask_handler.get(module.weight) if self.mask_handler is not None else torch.ones_like(module.weight))
            bias = None
            if module.bias is not None:
                bias = module.bias * (self.mask_handler.get(module.bias) if self.mask_handler is not None else torch.ones_like(module.bias))
            return F.conv2d(
                x, weight, bias,
                module.stride, module.padding,
                module.dilation, module.groups
            )

        elif isinstance(module, nn.Linear):
            weight = module.weight * (self.mask_handler.get(module.weight) if self.mask_handler is not None else torch.ones_like(module.weight))
            bias = None
            if module.bias is not None:
                bias = module.bias * (self.mask_handler.get(module.bias) if self.mask_handler is not None else torch.ones_like(module.bias))
            return F.linear(x, weight, bias)

        else:
            return module(x)

    def _make_layer(self, planes, num_blocks):
        layers = []
        for _ in range(num_blocks):
            layers.append(BasicBlock(self.in_planes, planes))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.apply_mask(self.bn1, self.apply_mask(self.conv1, x)))

        out = self.layer1(out)
        out = self.pool1(out)

        out = self.layer2(out)
        out = self.pool2(out)

        out = self.layer3(out)
        
        out = F.avg_pool2d(out, out.shape[2]) 
        out = out.view(out.size(0), -1)
        out = self.apply_mask(self.fc, out)  
        return out


# class MPM_BasicBlock(nn.Module):
#     def __init__(self, in_planes, planes):
#         super().__init__()
#         self.conv1 = MorphConv2d(in_planes, planes, kernel_size=(3,3), padding=1, bias=False)
#         self.linact1 = ConvLinAct(planes, method="simple")
#         # self.bn1 = nn.BatchNorm2d(planes)
#         self.bn1 = nn.Identity()
#         self.conv2 = MorphConv2d(planes, planes, kernel_size=(3,3), padding=1, bias=False)
#         self.linact2 = ConvLinAct(planes, method="simple")
#         # self.bn2 = nn.BatchNorm2d(planes)
#         self.bn2 = nn.Identity()

#         self.shortcut = nn.Sequential()
#         if in_planes != planes:
#             self.shortcut = MorphConv2d(in_planes, planes, kernel_size=(1,1), padding=0, bias=False)

#     def forward(self, x):
#         out = self.linact1(self.bn1(self.conv1(x)))
#         out = self.bn2(self.conv2(out))
#         out += self.shortcut(x)
#         out = self.linact2(out)
#         return out

class MPM_BasicBlock(nn.Module):
    def __init__(self, in_planes, planes):
        super().__init__()
        self.conv1 = MorphConv2d(in_planes, planes, kernel_size=(3,3), padding=1, bias=False)
        self.linact1 = ConvLinAct(planes)
        self.bn1 = nn.Identity()

        self.conv2 = MorphConv2d(planes, planes, kernel_size=(3,3), padding=1, bias=False)
        self.linact2 = ConvLinAct(planes)
        self.bn2 = nn.Identity()

        self.shortcut = None
        if in_planes != planes:
            self.shortcut = MorphConv2d(in_planes, planes, kernel_size=(1,1), padding=0, bias=False)

    def forward(self, inp):
        x, valid = inp

        out, valid_out = self.conv1((x, valid))
        out, valid_out = self.linact1((out, valid_out))

        out, valid_out = self.conv2((out, valid_out))

        if self.shortcut is None:
            sc, valid_sc = x, valid
        else:
            sc, valid_sc = self.shortcut((x, valid))

        combined_valid = valid_out | valid_sc
        out = torch.where(valid_out, out, torch.zeros_like(out)) + torch.where(valid_sc, sc, torch.zeros_like(sc))

        out, combined_valid = self.linact2((out, combined_valid))
        return out, combined_valid

class MPM_ResNet20(nn.Module):
    def __init__(self, in_channels=3, num_blocks=[3,3,3], num_classes=10):
        super().__init__()
        self.in_planes = 16

        self.conv1 = MorphConv2d(in_channels, 16, kernel_size=(3, 3), padding=1, bias=False)
        self.linact1 = ConvLinAct(16, method = "simple")
        # self.bn1 = nn.BatchNorm2d(16)
        self.bn1 = nn.Identity()

        self.layer1 = self._make_layer(16, num_blocks[0])
        self.pool1 = nn.MaxPool2d(2, 2)

        self.layer2 = self._make_layer(32, num_blocks[1])
        self.pool2 = nn.MaxPool2d(2, 2)

        self.layer3 = self._make_layer(64, num_blocks[2])
        
        self.fc = nn.Linear(64, num_classes)

        self.mask_handler = None

        self.state = "normal"

    def count_params(self):
        total = sum(p.numel() for p in self.parameters())
        active = 0
        for name, p in self.named_parameters():
            mask = self.mask_handler.get(p) if self.mask_handler is not None else torch.ones_like(p)
            print(f"{name}: Initial params = {p.numel()}, Post-pruning active params = {mask.sum().item()}")
            active += mask.sum().item()
        return total, active

    def set_state(self, state):
        self.state = state
        for m in self.modules():
            if isinstance(m, MorphConv2d):
                m.state = state

    def _make_layer(self, planes, num_blocks):
        layers = []
        for _ in range(num_blocks):
            layers.append(MPM_BasicBlock(self.in_planes, planes))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def pool_valid(self, x, valid, pool):
        x = pool(torch.where(valid, x, torch.full_like(x, float("-inf"))))
        valid = F.max_pool2d(valid.float(), kernel_size=pool.kernel_size, stride=pool.stride) > 0
        x = torch.where(valid, x, torch.zeros_like(x))
        return x, valid

    def forward(self, x):
        valid = torch.ones_like(x, dtype=torch.bool)

        x, valid = self.conv1((x, valid))
        x, valid = self.linact1((x, valid))

        x, valid = self.layer1((x, valid))
        x, valid = self.pool_valid(x, valid, self.pool1)

        x, valid = self.layer2((x, valid))
        x, valid = self.pool_valid(x, valid, self.pool2)

        x, valid = self.layer3((x, valid))

        x = torch.where(valid, x, torch.zeros_like(x))
        x = F.avg_pool2d(x, x.shape[2])
        x = x.view(x.size(0), -1)

        return self.fc(x)

    # def forward(self, x):
    #     out = self.linact1(self.bn1(self.conv1(x)))

    #     out = self.layer1(out)
    #     out = self.pool1(out)

    #     out = self.layer2(out)
    #     out = self.pool2(out)

    #     out = self.layer3(out)
        
    #     out = F.avg_pool2d(out, out.shape[2]) 
    #     out = out.view(out.size(0), -1)
    #     out = self.fc(out)
    #     return out  

Using /home/kfot/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module maxplus_conv2d, skipping build step...
Loading extension module maxplus_conv2d...


In [8]:
# Confirm that the CUDA MorphConv2D module has been loaded properly. Otherwise, clear 
# torch.extensions cache

input = torch.zeros((1, 1, 3, 3), requires_grad=True, device=device)
weight = torch.zeros((1, 1, 3, 3), requires_grad=True, device=device)
bias = torch.zeros((1,), requires_grad=True, device=device)

# print(input, weight, bias)

out = maxplus_conv2d_wrapper(input, weight, bias, 1, 0)

print(input, weight, bias)
print(out)

loss = torch.sum(out)
print(loss)

loss.backward()

print(input.grad)
print(weight.grad)
print(bias.grad)

tensor([[[[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]], device='cuda:0', requires_grad=True) tensor([[[[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]], device='cuda:0', requires_grad=True) tensor([0.], device='cuda:0', requires_grad=True)
tensor([[[[0.]]]], device='cuda:0', grad_fn=<MaxPlusConv2dFunctionBackward>)
tensor(0., device='cuda:0', grad_fn=<SumBackward0>)
tensor([[[[1., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]], device='cuda:0')
tensor([[[[1., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]], device='cuda:0')
tensor([0.], device='cuda:0')


In [9]:
def train(model, criterion, optimizer, train_loader, val_loader, num_epochs=50, return_list=False):
    # Training and validation loop
    best_val_accuracy = 0.0
    best_model = None

    train_list = []
    val_list = []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Used during development to ensure inactive parameters truly are inactive
            # with torch.no_grad():
            #     if model.mask_handler is not None:
            #         for name, p in model.named_parameters():
            #             mask = model.mask_handler.get(p)
            #             p.mul_(mask)

        # Validation phase
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        train_accuracy = 100 * correct / total
        train_list.append(train_accuracy)
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_accuracy = 100 * correct / total
        val_list.append(val_accuracy)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Train Accuracy: {train_accuracy:.2f}%, Validation Accuracy: {val_accuracy:.2f}%")

        # Save best model based on validation accuracy
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model = copy.deepcopy(model)
            if model.mask_handler is not None:
                mask_dict = model.mask_handler.export()

                best_model.mask_handler = MaskHandler()
                best_model.mask_handler.load(best_model, mask_dict)

    if return_list:
        return best_model, train_list, val_list
    else:
        return best_model

def test(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy on the test set: {accuracy:.2f}%')

In [10]:
# Load and preprocess MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

full_train_dataset = torchvision.datasets.FashionMNIST(root='../data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.FashionMNIST(root='../data', train=False, transform=transform, download=True)

# Split train dataset into training and validation sets
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(seed))

In [11]:
# Data loaders
def make_loader(dataset, batch_size, shuffle, num_workers=0):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        worker_init_fn=seed_worker,
        generator=generator,
        pin_memory=True
    )

In [12]:
def run_experiment(model, train_dataset, val_dataset, test_dataset, pruning_ratio=0.5, num_batches=10, method="snip"):
    # Reset randomness
    set_seed(seed)
    train_loader = make_loader(train_dataset, batch_size=64, shuffle=True)
    val_loader = make_loader(val_dataset, batch_size=64, shuffle=False)
    test_loader = make_loader(test_dataset, batch_size=64, shuffle=False)

    model = model[0](**model[1]).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model = train(model, criterion, optimizer, train_loader, val_loader, num_epochs=1)

    pruner = Pruner(model=model)
    params_initial, params_active = pruner.prune(train_loader, pruning_ratio, num_batches=num_batches, method=method)

    print(f"Initial number of parameters: {params_initial}")
    print(f"Total number of active parameters after pruning: {params_active}")

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model = train(model, criterion, optimizer, train_loader, val_loader)

    print(f"Total number of active parameters on testing: {model.count_params()[1]}")
    test(model, test_loader)

In [11]:
run_experiment([ResNet20, {"in_channels": 1}], 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.95, 
               num_batches=10, 
               method="snip"
)

Seed set to: 1874033229


Epoch [1/1], Loss: 0.2497, Train Accuracy: 86.31%, Validation Accuracy: 86.22%
conv1.weight: Initial params = 144, Post-pruning active params = 136
bn1.weight: Initial params = 16, Post-pruning active params = 16
bn1.bias: Initial params = 16, Post-pruning active params = 10
layer1.0.conv1.weight: Initial params = 2304, Post-pruning active params = 1343
layer1.0.bn1.weight: Initial params = 16, Post-pruning active params = 16
layer1.0.bn1.bias: Initial params = 16, Post-pruning active params = 4
layer1.0.conv2.weight: Initial params = 2304, Post-pruning active params = 1343
layer1.0.bn2.weight: Initial params = 16, Post-pruning active params = 16
layer1.0.bn2.bias: Initial params = 16, Post-pruning active params = 0
layer1.1.conv1.weight: Initial params = 2304, Post-pruning active params = 946
layer1.1.bn1.weight: Initial params = 16, Post-pruning active params = 16
layer1.1.bn1.bias: Initial params = 16, Post-pruning active params = 0
layer1.1.conv2.weight: Initial params = 2304, Post

In [12]:
run_experiment([ResNet20, {"in_channels": 1}], 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.975, 
               num_batches=10, 
               method="snip"
)

Seed set to: 1874033229


Epoch [1/1], Loss: 0.2497, Train Accuracy: 86.31%, Validation Accuracy: 86.22%
conv1.weight: Initial params = 144, Post-pruning active params = 136
bn1.weight: Initial params = 16, Post-pruning active params = 16
bn1.bias: Initial params = 16, Post-pruning active params = 6
layer1.0.conv1.weight: Initial params = 2304, Post-pruning active params = 914
layer1.0.bn1.weight: Initial params = 16, Post-pruning active params = 16
layer1.0.bn1.bias: Initial params = 16, Post-pruning active params = 2
layer1.0.conv2.weight: Initial params = 2304, Post-pruning active params = 876
layer1.0.bn2.weight: Initial params = 16, Post-pruning active params = 16
layer1.0.bn2.bias: Initial params = 16, Post-pruning active params = 0
layer1.1.conv1.weight: Initial params = 2304, Post-pruning active params = 446
layer1.1.bn1.weight: Initial params = 16, Post-pruning active params = 16
layer1.1.bn1.bias: Initial params = 16, Post-pruning active params = 0
layer1.1.conv2.weight: Initial params = 2304, Post-pr

In [16]:
run_experiment([MPM_ResNet20, {"in_channels": 1}], 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.95, 
               num_batches=10, 
               method="snip"
)

Seed set to: 1874033229
Epoch [1/1], Loss: 0.9756, Train Accuracy: 63.79%, Validation Accuracy: 63.96%
conv1.weight: Initial params = 144, Post-pruning active params = 93
linact1.a: Initial params = 144, Post-pruning active params = 71
layer1.0.conv1.weight: Initial params = 2304, Post-pruning active params = 179
layer1.0.linact1.a: Initial params = 144, Post-pruning active params = 46
layer1.0.conv2.weight: Initial params = 2304, Post-pruning active params = 143
layer1.0.linact2.a: Initial params = 144, Post-pruning active params = 119
layer1.1.conv1.weight: Initial params = 2304, Post-pruning active params = 145
layer1.1.linact1.a: Initial params = 144, Post-pruning active params = 36
layer1.1.conv2.weight: Initial params = 2304, Post-pruning active params = 138
layer1.1.linact2.a: Initial params = 144, Post-pruning active params = 104
layer1.2.conv1.weight: Initial params = 2304, Post-pruning active params = 156
layer1.2.linact1.a: Initial params = 144, Post-pruning active params = 

In [17]:
run_experiment([MPM_ResNet20, {"in_channels": 1}], 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.975, 
               num_batches=10, 
               method="snip"
)

Seed set to: 1874033229
Epoch [1/1], Loss: 0.9755, Train Accuracy: 63.79%, Validation Accuracy: 63.95%
conv1.weight: Initial params = 144, Post-pruning active params = 72
linact1.a: Initial params = 144, Post-pruning active params = 30
layer1.0.conv1.weight: Initial params = 2304, Post-pruning active params = 111
layer1.0.linact1.a: Initial params = 144, Post-pruning active params = 16
layer1.0.conv2.weight: Initial params = 2304, Post-pruning active params = 117
layer1.0.linact2.a: Initial params = 144, Post-pruning active params = 26
layer1.1.conv1.weight: Initial params = 2304, Post-pruning active params = 99
layer1.1.linact1.a: Initial params = 144, Post-pruning active params = 16
layer1.1.conv2.weight: Initial params = 2304, Post-pruning active params = 107
layer1.1.linact2.a: Initial params = 144, Post-pruning active params = 35
layer1.2.conv1.weight: Initial params = 2304, Post-pruning active params = 93
layer1.2.linact1.a: Initial params = 144, Post-pruning active params = 16
l

In [18]:
run_experiment([MPM_ResNet20, {"in_channels": 1}], 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.95, 
               num_batches=10, 
               method="snip2"
)

Seed set to: 1874033229


Epoch [1/1], Loss: 0.9754, Train Accuracy: 63.79%, Validation Accuracy: 63.97%
conv1.weight: Initial params = 144, Post-pruning active params = 86
linact1.a: Initial params = 144, Post-pruning active params = 144
layer1.0.conv1.weight: Initial params = 2304, Post-pruning active params = 144
layer1.0.linact1.a: Initial params = 144, Post-pruning active params = 144
layer1.0.conv2.weight: Initial params = 2304, Post-pruning active params = 129
layer1.0.linact2.a: Initial params = 144, Post-pruning active params = 144
layer1.1.conv1.weight: Initial params = 2304, Post-pruning active params = 121
layer1.1.linact1.a: Initial params = 144, Post-pruning active params = 144
layer1.1.conv2.weight: Initial params = 2304, Post-pruning active params = 122
layer1.1.linact2.a: Initial params = 144, Post-pruning active params = 144
layer1.2.conv1.weight: Initial params = 2304, Post-pruning active params = 126
layer1.2.linact1.a: Initial params = 144, Post-pruning active params = 144
layer1.2.conv2.we

In [19]:
run_experiment([MPM_ResNet20, {"in_channels": 1}], 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.975, 
               num_batches=10, 
               method="snip2"
)

Seed set to: 1874033229
Epoch [1/1], Loss: 0.9754, Train Accuracy: 63.78%, Validation Accuracy: 63.98%
conv1.weight: Initial params = 144, Post-pruning active params = 40
linact1.a: Initial params = 144, Post-pruning active params = 144
layer1.0.conv1.weight: Initial params = 2304, Post-pruning active params = 47
layer1.0.linact1.a: Initial params = 144, Post-pruning active params = 144
layer1.0.conv2.weight: Initial params = 2304, Post-pruning active params = 54
layer1.0.linact2.a: Initial params = 144, Post-pruning active params = 144
layer1.1.conv1.weight: Initial params = 2304, Post-pruning active params = 40
layer1.1.linact1.a: Initial params = 144, Post-pruning active params = 144
layer1.1.conv2.weight: Initial params = 2304, Post-pruning active params = 48
layer1.1.linact2.a: Initial params = 144, Post-pruning active params = 144
layer1.2.conv1.weight: Initial params = 2304, Post-pruning active params = 25
layer1.2.linact1.a: Initial params = 144, Post-pruning active params = 14

In [20]:
# Load and preprocess CIFAR dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))])

full_train_dataset = torchvision.datasets.CIFAR10(root='../data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.CIFAR10(root='../data', train=False, transform=transform, download=True)

# Split train dataset into training and validation sets
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(seed))

Files already downloaded and verified
Files already downloaded and verified


In [16]:
run_experiment([ResNet20, {"in_channels": 3}], 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.95, 
               num_batches=10, 
               method="snip"
)

Seed set to: 1874033229
Epoch [1/1], Loss: 0.8469, Train Accuracy: 58.55%, Validation Accuracy: 56.47%
conv1.weight: Initial params = 432, Post-pruning active params = 394
bn1.weight: Initial params = 16, Post-pruning active params = 16
bn1.bias: Initial params = 16, Post-pruning active params = 12
layer1.0.conv1.weight: Initial params = 2304, Post-pruning active params = 1474
layer1.0.bn1.weight: Initial params = 16, Post-pruning active params = 16
layer1.0.bn1.bias: Initial params = 16, Post-pruning active params = 6
layer1.0.conv2.weight: Initial params = 2304, Post-pruning active params = 1381
layer1.0.bn2.weight: Initial params = 16, Post-pruning active params = 16
layer1.0.bn2.bias: Initial params = 16, Post-pruning active params = 2
layer1.1.conv1.weight: Initial params = 2304, Post-pruning active params = 1213
layer1.1.bn1.weight: Initial params = 16, Post-pruning active params = 16
layer1.1.bn1.bias: Initial params = 16, Post-pruning active params = 2
layer1.1.conv2.weight: In

In [17]:
run_experiment([ResNet20, {"in_channels": 3}], 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.975, 
               num_batches=10, 
               method="snip"
)

Seed set to: 1874033229
Epoch [1/1], Loss: 0.8469, Train Accuracy: 58.55%, Validation Accuracy: 56.47%
conv1.weight: Initial params = 432, Post-pruning active params = 376
bn1.weight: Initial params = 16, Post-pruning active params = 16
bn1.bias: Initial params = 16, Post-pruning active params = 7
layer1.0.conv1.weight: Initial params = 2304, Post-pruning active params = 898
layer1.0.bn1.weight: Initial params = 16, Post-pruning active params = 16
layer1.0.bn1.bias: Initial params = 16, Post-pruning active params = 3
layer1.0.conv2.weight: Initial params = 2304, Post-pruning active params = 770
layer1.0.bn2.weight: Initial params = 16, Post-pruning active params = 16
layer1.0.bn2.bias: Initial params = 16, Post-pruning active params = 0
layer1.1.conv1.weight: Initial params = 2304, Post-pruning active params = 688
layer1.1.bn1.weight: Initial params = 16, Post-pruning active params = 16
layer1.1.bn1.bias: Initial params = 16, Post-pruning active params = 1
layer1.1.conv2.weight: Initia

In [21]:
run_experiment([MPM_ResNet20, {"in_channels": 3}], 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.95, 
               num_batches=10, 
               method="snip"
)

Seed set to: 1874033229


Epoch [1/1], Loss: 2.2012, Train Accuracy: 21.94%, Validation Accuracy: 22.24%
conv1.weight: Initial params = 432, Post-pruning active params = 129
linact1.a: Initial params = 144, Post-pruning active params = 24
layer1.0.conv1.weight: Initial params = 2304, Post-pruning active params = 138
layer1.0.linact1.a: Initial params = 144, Post-pruning active params = 16
layer1.0.conv2.weight: Initial params = 2304, Post-pruning active params = 121
layer1.0.linact2.a: Initial params = 144, Post-pruning active params = 16
layer1.1.conv1.weight: Initial params = 2304, Post-pruning active params = 125
layer1.1.linact1.a: Initial params = 144, Post-pruning active params = 16
layer1.1.conv2.weight: Initial params = 2304, Post-pruning active params = 133
layer1.1.linact2.a: Initial params = 144, Post-pruning active params = 22
layer1.2.conv1.weight: Initial params = 2304, Post-pruning active params = 128
layer1.2.linact1.a: Initial params = 144, Post-pruning active params = 16
layer1.2.conv2.weight:

In [22]:
run_experiment([MPM_ResNet20, {"in_channels": 3}], 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.95, 
               num_batches=10, 
               method="snip2"
)

Seed set to: 1874033229
Epoch [1/1], Loss: 2.2012, Train Accuracy: 21.92%, Validation Accuracy: 22.28%
conv1.weight: Initial params = 432, Post-pruning active params = 113
linact1.a: Initial params = 144, Post-pruning active params = 144
layer1.0.conv1.weight: Initial params = 2304, Post-pruning active params = 114
layer1.0.linact1.a: Initial params = 144, Post-pruning active params = 144
layer1.0.conv2.weight: Initial params = 2304, Post-pruning active params = 100
layer1.0.linact2.a: Initial params = 144, Post-pruning active params = 144
layer1.1.conv1.weight: Initial params = 2304, Post-pruning active params = 98
layer1.1.linact1.a: Initial params = 144, Post-pruning active params = 144
layer1.1.conv2.weight: Initial params = 2304, Post-pruning active params = 104
layer1.1.linact2.a: Initial params = 144, Post-pruning active params = 144
layer1.2.conv1.weight: Initial params = 2304, Post-pruning active params = 95
layer1.2.linact1.a: Initial params = 144, Post-pruning active params 

In [23]:
run_experiment([MPM_ResNet20, {"in_channels": 3}], 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.975, 
               num_batches=10, 
               method="snip2"
)

Seed set to: 1874033229
Epoch [1/1], Loss: 2.2011, Train Accuracy: 21.91%, Validation Accuracy: 22.19%
conv1.weight: Initial params = 432, Post-pruning active params = 44
linact1.a: Initial params = 144, Post-pruning active params = 144
layer1.0.conv1.weight: Initial params = 2304, Post-pruning active params = 16
layer1.0.linact1.a: Initial params = 144, Post-pruning active params = 144
layer1.0.conv2.weight: Initial params = 2304, Post-pruning active params = 34
layer1.0.linact2.a: Initial params = 144, Post-pruning active params = 144
layer1.1.conv1.weight: Initial params = 2304, Post-pruning active params = 12
layer1.1.linact1.a: Initial params = 144, Post-pruning active params = 144
layer1.1.conv2.weight: Initial params = 2304, Post-pruning active params = 14
layer1.1.linact2.a: Initial params = 144, Post-pruning active params = 144
layer1.2.conv1.weight: Initial params = 2304, Post-pruning active params = 4
layer1.2.linact1.a: Initial params = 144, Post-pruning active params = 144